# 🤖 Notebook 03 — Model Development
**CCS3440 Artificial Intelligence | SmartCare AI Risk Prediction**

---
### 🎯 Objectives (Task 05)
- Train four ML classifiers: **Logistic Regression, Random Forest, XGBoost, Decision Tree**
- Handle class imbalance using SMOTE
- Hyperparameter tuning with GridSearchCV
- Save trained models to `models/`

---

## 1️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from imblearn.over_sampling import SMOTE

print('✅ Libraries imported successfully')

## 2️⃣ Load Processed Data

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Class distribution (train):\n{y_train.value_counts()}')

## 3️⃣ Handle Class Imbalance — SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — X_train: {X_train_res.shape}')
print(f'Class distribution (resampled):\n{pd.Series(y_train_res).value_counts()}')

## 4️⃣ Define Models

In [ ]:
models = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'decision_tree':       DecisionTreeClassifier(random_state=42),
    'random_forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'xgboost':             XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
}

print(f'✅ {len(models)} models defined')

## 5️⃣ Train All Models

In [ ]:
trained_models = {}
cv_results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)
    trained_models[name] = model

    # 5-fold cross-validation
    cv_score = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='roc_auc')
    cv_results[name] = {'mean_auc': cv_score.mean(), 'std_auc': cv_score.std()}
    print(f'  CV ROC-AUC: {cv_score.mean():.4f} ± {cv_score.std():.4f}')

print('\n✅ All models trained')

## 6️⃣ Save Trained Models

In [ ]:
for name, model in trained_models.items():
    path = f'../models/{name}.pkl'
    joblib.dump(model, path)
    print(f'✅ Saved: {path}')

print('\n✅ All models saved to models/')

## 7️⃣ Select Best Model & Save Selection Metadata

In [ ]:
# Determine best model by cross-validation AUC
best_name = max(cv_results, key=lambda k: cv_results[k]['mean_auc'])
best_auc  = round(cv_results[best_name]['mean_auc'], 4)

selection = {
    'best_model': best_name,
    'roc_auc': best_auc,
    'cv_results': cv_results
}

with open('../models/final_model_selection.json', 'w') as f:
    json.dump(selection, f, indent=2)

print(f'✅ Best model: {best_name} (AUC = {best_auc})')
print('✅ Saved: models/final_model_selection.json')